# 09 — Static Factors, Non-Switching Factor Dynamics and the Price-Only Factor Benchmark (framework v3.2)

**Scope.** Framework v3.2 §21 steps 3–6, i.e. the nested structural sequence §6.12 (A)–(C):

| Part | v3.2 | Model |
|---|---|---|
| A | §21 step 3, §6.12 (A) | Static factors: training-only per-family PCA, deterministic signs, **masked / observed-coordinate scoring** (§6.7), coverage and loading stability (§6.8) |
| B | §21 step 4, §6.12 (B) | Non-switching factor-augmented price baseline: $z_{t+1} = \mu + \beta_D D_t + \gamma^\top F^{dir}_t + \rho^\top L_t + \sigma\xi_{t+1}$ |
| C | §21 step 5, §6.12 (B) | Non-switching dynamic factor model $F_t = c + A F_{t-1} + \eta_t$ |
| D | §21 step 6, §6.12 (C), §6.13 | Price-only factor-augmented switching benchmark ($\omega_F = 0$), HMM first, then HSMM; next to the Notebook 08 benchmark |
| E | §7 | States of the benchmark |
| F | §6.10 | Conditional sign-destruction null (one of the two required nulls; **not confirmatory** on its own) |

The joint SDF-HMM with regime-specific $c_R, A_R, \Sigma_R$ and tempered factor evidence $\omega_F$ (§6.5, §21 step 7) is the next module.

**Comparison unit (§19.1).** Every out-of-sample comparison is paired by session on identical test bars: per-session difference in next-return log score, mean and median, share of sessions won, whole-session bootstrap 95% interval, leave-one-session-out range. The per-bar Newey–West $t$ is secondary. Test sessions are the second half of the sample (about 7 sessions): exploratory evidence (§6.14).

**Status: exploratory** (≈ 15 sessions; v3.2 gates: 30–60 days for factor diagnostics, 60–90 days for exploratory switching-factor fits).

In [17]:
from pathlib import Path
import os
import sys
import time
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

load_dotenv()
DB_PATH = Path(os.environ["DERIBIT_DATA_ROOT"]).expanduser().resolve() / "database" / "research.duckdb"
con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute("SET TimeZone = 'UTC'")

PROJECT_ROOT = Path(os.getenv("DERIBIT_PROJECT_ROOT", Path.cwd())).resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "data_engineering").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "data_engineering").exists():
    raise RuntimeError("Repository not found: start Jupyter inside the repository or set DERIBIT_PROJECT_ROOT.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
for _m in [m for m in sys.modules if m.split(".")[0] == "data_engineering"]:
    del sys.modules[_m]

from data_engineering.features import regime_inputs as ri
from data_engineering.modeling.clock_baseline import ClockBaselineSet, FeatureSpec
from data_engineering.modeling import factors as fx
from data_engineering.modeling import dfm
from data_engineering.modeling.hsmm.engine import HSMMSpec, fit, episodes, predictive_scores, ENGINE_VERSION, USE_NUMBA
from data_engineering.modeling.hsmm.design import build_design
from data_engineering.modeling.hsmm.selection import select_K
from data_engineering.modeling.hsmm.session_eval import paired_sessions
from data_engineering.modeling.hsmm.null import simulate_returns, p_value
if tuple(int(x) for x in ri.REGIME_INPUT_VERSION.split(".")) < (0, 3) or ENGINE_VERSION < "0.1.2" or fx.FACTOR_VERSION < "0.2":
    raise RuntimeError("Old src modules: git pull and restart the kernel.")

CONTRACT_ASSETS = ["AVAX", "BTC", "ETH", "HYPE", "SOL", "XRP"]
K_MAX = int(os.getenv("NB09_K_MAX", "4"))
NULL_PATHS = int(os.getenv("NB09_NULL_PATHS", "19"))
NULL_ASSETS = os.getenv("NB09_NULL_ASSETS", "BTC,XRP,ETH,SOL").split(",")
N_BOOT = int(os.getenv("NB09_BOOT", "2000"))
FIT_KW = {"n_init": 2, "max_iter": 200, "tol": 1e-5}
LAGS = ["z1", "z2", "z3"]
print(f"src {PROJECT_ROOT / 'src'} | engine {ENGINE_VERSION} | factors {fx.FACTOR_VERSION} | numba {'on' if USE_NUMBA else 'OFF'} | "
      f"K = 1..{K_MAX} | null {NULL_PATHS} paths on {NULL_ASSETS}")


def compare(scores: dict, sessions: np.ndarray, pairs: list) -> pd.DataFrame:
    rows = []
    for a, b in pairs:
        if a in scores and b in scores:
            rows.append({"model": a, "vs": b, **paired_sessions(scores[a], scores[b], sessions, n_boot=N_BOOT)})
    return pd.DataFrame(rows)


SHOW = ["model", "vs", "n_sessions", "mean_session_diff", "boot_lo", "boot_hi", "median_session_diff", "sessions_won",
        "loso_min", "loso_max", "per_bar_x1000", "per_bar_nw_t"]

src D:\Projects\deribit_market_data\src | engine 0.1.2 | factors 0.2 | numba on | K = 1..4 | null 19 paths on ['BTC', 'XRP', 'ETH', 'SOL']


## 1. Load, sessions and clock factor for returns

In [18]:
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])
base_cols = ["state_id", "state_time", "underlying", "session_id", "return_5m", "futures_index_price", "is_weekend",
             "in_settlement_window", "fit_ok", "option_atm_quote_ok",
             "vwap", "volume_usd", "trade_count", "futures_tradable_mid_price", "perp_integrity_ok",
             "known_exchange_event", "integrity_flag_trades_outside_bbo"]
factor_cols = sorted({fd.column for fam in fx.FAMILIES for fd in fam.features} |
                     {"trade_flow_imbalance", "book_twa_obi_20", "book_twa_bid_depth_20", "book_twa_ask_depth_20",
                      "option_iv_atm_forward", "realized_vol_1h_annualized"})
want = [c for c in dict.fromkeys(base_cols + factor_cols) if c in cols]
print("Not in time_series_feature_state (skipped):", sorted(set(base_cols + factor_cols) - cols - {
    "tfi_30m", "obi_20_30m", "book_depth_20_usd", "atm_iv_quote", "iv_minus_rv"}))
df = con.execute(f"SELECT {', '.join(want)} FROM time_series_feature_state ORDER BY underlying, state_time").df()
df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
for c in ["is_weekend", "in_settlement_window", "fit_ok", "option_atm_quote_ok", "perp_integrity_ok",
          "known_exchange_event", "integrity_flag_trades_outside_bbo"]:
    if c in df:
        s = df[c]
        if s.dtype == object:
            s = s.map({"True": True, "False": False, True: True, False: False})
        df[c] = s.fillna(False).astype(bool)
df = df[df["underlying"].isin(CONTRACT_ASSETS)].reset_index(drop=True)
df["r"] = pd.to_numeric(df["return_5m"], errors="coerce")
df["session_date"] = ri.session_key(df["state_time"])
dates = sorted(df.loc[df["r"].notna(), "session_date"].unique())
TRAIN_DATES = set(dates[: len(dates) // 2])
train_mask = df["session_date"].isin(TRAIN_DATES)
clock = ClockBaselineSet.fit(df[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]], train_mask,
                             specs=(FeatureSpec("return_5m", "return_5m", transform="log_abs", scale="none"),))
df["f_clock"] = clock.transform(df[["state_id", "state_time", "underlying", "is_weekend", "return_5m"]])[
    "return_5m__clock_vol_factor"].to_numpy()
df["f_clock"] = df["f_clock"].fillna(1.0)
print(f"Rows {len(df):,} | assets {sorted(df.underlying.unique())} | sessions {len(dates)} | "
      f"train {min(TRAIN_DATES)} → {max(TRAIN_DATES)} | test {min(set(dates) - TRAIN_DATES)} → {max(dates)}")

Not in time_series_feature_state (skipped): []
Rows 29,376 | assets ['AVAX', 'BTC', 'ETH', 'HYPE', 'SOL', 'XRP'] | sessions 15 | train 2026-09-22 → 2026-09-28 | test 2026-09-29 → 2026-10-06


# Part A — Static factors (§21 step 3)

Each feature is clock-adjusted (Fourier baseline with weekend terms, training only), standardized with training moments and winsorized at ±5. Each family's loadings are the leading eigenvector of the **pairwise-complete** training correlation matrix (valid observations only), signed by the family's anchor feature. Scores use the **observed coordinates only**,

$$
f_{g,t} = \frac{\sum_{j \in O_t} w_{g,j}\,\tilde x_{j,t}}{\sum_{j \in O_t} w_{g,j}^2}
$$

and a family with no observed feature at $t$ is **unavailable** — never a neutral zero (§6.7). Eligibility (§6.8): coverage ≥ `fx.MIN_COVERAGE` and loading stability between the two halves of the training window ≥ `fx.MIN_STABILITY` (single-feature families need coverage only). Ineligible families are excluded from Parts B–F.

In [19]:
t0 = time.perf_counter()
FS = fx.build_factors(df, train_mask)
print(f"Factors built in {time.perf_counter() - t0:.0f}s | missing raw columns: {FS.missing_columns} | "
      f"clock fits skipped: {len(FS.clock_skipped)} | thresholds: coverage >= {fx.MIN_COVERAGE}, stability >= {fx.MIN_STABILITY}")
for col, title in [("explained_pc1", "Explained share of the first component"), ("coverage", "Coverage (factor available)"),
                   ("loading_stability", "Loading stability, |cos| between training halves"), ("eligible", "Eligible")]:
    print(title + ":")
    display(FS.summary.pivot(index=["channel", "family"], columns="underlying", values=col).round(2))
print("Loadings:")
display(FS.loadings.pivot_table(index=["family", "feature"], columns="underlying", values="loading").round(2))
corrs = [FS.factors[train_mask & (df["underlying"] == a)].corr() for a in sorted(df["underlying"].unique())]
print("Median (over assets) pairwise correlation between factors, training rows (available rows only):")
display(pd.concat(corrs).groupby(level=0).median().reindex(FS.factors.columns)[FS.factors.columns].round(2))

ELIG = {a: set("F_" + FS.summary[(FS.summary.underlying == a) & FS.summary.eligible].family) for a in CONTRACT_ASSETS}
DIR = {a: [c for c in FS.directional().columns if c in ELIG.get(a, set())] for a in CONTRACT_ASSETS}
STATE = {a: [c for c in FS.state().columns if c in ELIG.get(a, set())] for a in CONTRACT_ASSETS}
display(pd.DataFrame({"directional": {a: [c[2:] for c in DIR[a]] for a in DIR},
                      "state": {a: [c[2:] for c in STATE[a]] for a in STATE}}))

Factors built in 2s | missing raw columns: [] | clock fits skipped: 0 | thresholds: coverage >= 0.5, stability >= 0.8
Explained share of the first component:


underlying                AVAX   BTC   ETH  HYPE   SOL   XRP
channel     family                                          
directional basis         1.00  1.00  1.00  1.00  1.00  1.00
            book          0.50  0.59  0.60  0.40  0.61  0.51
            flow          0.71  0.70  0.72  0.74  0.72  0.70
            skew          0.73  0.55  0.58  0.69  0.70  0.66
state       activity      0.69  0.57  0.52  0.64  0.53  0.66
            liquidity     0.53  0.73  0.64  0.64  0.55  0.66
            option_level  0.52  0.45  0.50  0.41  0.52  0.40
            volatility    0.67  0.72  0.73  0.62  0.70  0.69

Coverage (factor available):


underlying                AVAX   BTC   ETH  HYPE   SOL   XRP
channel     family                                          
directional basis         0.86  0.86  0.86  0.86  0.86  0.86
            book          0.80  0.80  0.80  0.80  0.80  0.80
            flow          0.85  0.88  0.88  0.80  0.88  0.86
            skew          0.74  0.56  0.63  0.65  0.60  0.58
state       activity      0.88  0.88  0.88  0.88  0.88  0.88
            liquidity     0.80  0.80  0.80  0.80  0.80  0.80
            option_level  0.85  0.85  0.85  0.84  0.81  0.79
            volatility    0.86  0.87  0.87  0.85  0.87  0.87

Loading stability, |cos| between training halves:


underlying                AVAX   BTC   ETH  HYPE   SOL   XRP
channel     family                                          
directional basis          NaN   NaN   NaN   NaN   NaN   NaN
            book          1.00  1.00  1.00  0.96  1.00  0.99
            flow          1.00  1.00  1.00  1.00  1.00  1.00
            skew          1.00  1.00  1.00  1.00  1.00  1.00
state       activity      0.99  0.69  0.68  1.00  0.73  1.00
            liquidity     1.00  1.00  1.00  1.00  1.00  1.00
            option_level  1.00  0.71  0.96  0.56  0.99  0.79
            volatility    1.00  1.00  1.00  1.00  1.00  1.00

... (3 outputs left out)

underlying                     AVAX   BTC   ETH  HYPE   SOL   XRP
family       feature                                             
activity     log_book_updates  0.45  0.18  0.09  0.35  0.36  0.42
             log_trades        0.64  0.70  0.70  0.66  0.67  0.65
             log_volume        0.62  0.69  0.70  0.67  0.65  0.63
basis        basis_bps         1.00  1.00  1.00  1.00  1.00  1.00
book         microprice_bps   -0.04 -0.05  0.02 -0.05  0.05  0.10
             obi_10            0.50  0.55  0.55  0.27  0.53  0.53
             obi_20            0.61  0.54  0.55  0.65  0.49  0.54
             obi_20_30m        0.60  0.46  0.49  0.64  0.46  0.50
             obi_5            -0.09  0.43  0.39  0.30  0.51  0.40
flow         tfi               0.71  0.71  0.71  0.71  0.71  0.71
             tfi_30m           0.71  0.71  0.71  0.71  0.71  0.71
liquidity    log_depth         0.71  0.71  0.71 -0.71 -0.71  0.71
             log_spread        0.71  0.71  0.71  0.71  0.71  0.71
option_level curvature         0.46 -0.50 -0.67 -0.15 -0.64 -0.36
             iv_minus_rv       0.59  0.52  0.38  0.71  0.31  0.59
             log_atm_iv        0.66  0.69  0.64  0.69  0.71  0.72
skew         skew              0.71  0.71  0.71  0.71  0.71  0.71
             skew_change_30m   0.71  0.71  0.71  0.71  0.71  0.71
volatility   log_range         0.45  0.50  0.51  0.33  0.51  0.47
             log_rv_1h         0.62  0.60  0.60  0.66  0.60  0.62
             log_rv_30m        0.64  0.62  0.62  0.67  0.62  0.63

Median (over assets) pairwise correlation between factors, training rows (available rows only):


,F_flow,F_book,F_basis,F_skew,F_activity,F_volatility,F_liquidity,F_option_level
F_flow,1.00,-0.06,0.00,0.05,-0.02,-0.00,-0.05,-0.02
F_book,-0.06,1.00,0.01,-0.04,-0.04,-0.04,0.02,-0.10
F_basis,0.00,0.01,1.00,0.10,0.04,0.06,-0.02,-0.15
F_skew,0.05,-0.04,0.10,1.00,0.12,0.05,-0.00,0.28
F_activity,-0.02,-0.04,0.04,0.12,1.00,0.51,0.07,0.15
F_volatility,-0.00,-0.04,0.06,0.05,0.51,1.00,0.26,-0.05
F_liquidity,-0.05,0.02,-0.02,-0.00,0.07,0.26,1.00,0.05
F_option_level,-0.02,-0.10,-0.15,0.28,0.15,-0.05,0.05,1.00


,directional,state
AVAX,"[flow, book, basis, skew]","[activity, volatility, liquidity, option_level]"
BTC,"[flow, book, basis, skew]","[volatility, liquidity]"
ETH,"[flow, book, basis, skew]","[volatility, liquidity, option_level]"
HYPE,"[flow, book, basis, skew]","[activity, volatility, liquidity]"
SOL,"[flow, book, basis, skew]","[volatility, liquidity, option_level]"
XRP,"[flow, book, basis, skew]","[activity, volatility, liquidity]"


## Designs

One design per asset: session-VWAP displacement, three shared return lags, and the asset's eligible directional factors as lagged regressors (known at $t$ for $z_{t+1}$). An unavailable factor's term is masked out of the mean for that bar (§6.7); its slope is estimated from the bars where it is available. All models of an asset share the same rows and test bars.

In [20]:
designs = {}
for asset, g in df.groupby("underlying"):
    idx = g.index
    g = g.reset_index(drop=True)
    E = FS.factors.loc[idx, DIR[asset]].reset_index(drop=True)
    d = build_design(g, g["f_clock"].to_numpy(), extra=E if len(E.columns) else None)
    tr = d.frame["session_date"].isin(TRAIN_DATES).to_numpy()
    te_rows = d.rows_of(~tr)
    designs[asset] = {"d": d, "g": g, "idx": idx, "tr": tr, "test_sessions": d.frame["session_date"].to_numpy()[te_rows]}
display(pd.DataFrame({a: {"rows": len(v["d"].frame), "segments": len(v["d"].starts),
                          "test_sessions": len(set(v["test_sessions"])), "test_bars": len(v["test_sessions"]),
                          **{f"avail_{c[2:]}": float(FS.factors.loc[v["idx"], c].notna().mean()) for c in DIR[a]}}
                      for a, v in designs.items()}).T)

,rows,segments,test_sessions,test_bars,avail_flow,avail_book,avail_basis,avail_skew
AVAX,3393.0,34.0,8.0,1740.0,0.854575,0.804534,0.858456,0.740809
BTC,3396.0,30.0,8.0,1742.0,0.876021,0.804739,0.858864,0.556985
ETH,3396.0,31.0,8.0,1742.0,0.876021,0.804739,0.858864,0.625408
HYPE,3396.0,31.0,8.0,1743.0,0.797590,0.804534,0.858456,0.654208
SOL,3395.0,32.0,8.0,1742.0,0.875817,0.804330,0.858456,0.601307
XRP,3395.0,31.0,8.0,1742.0,0.861520,0.804534,0.858456,0.582925


# Part B — Non-switching factor-augmented price baseline (§21 step 4)

$K = 1$: the Notebook 08 structure without regimes (**Base**: lags + $D$), plus one directional family at a time, plus all eligible directional families. Fitted on the training sessions, scored on the test sessions; paired by session against Base.

In [21]:
B_rows, SCORES = [], {}
for asset, v in designs.items():
    d, tr = v["d"], v["tr"]
    models = [("Base K1", ["const", "D"])] + [(f"+{c[2:]} K1", ["const", "D", c]) for c in DIR[asset]]
    if len(DIR[asset]) > 1:
        models.append(("+dir K1", ["const", "D"] + DIR[asset]))
    sc = {}
    for name, xs in models:
        res = fit(d.data(xs=xs, xc=LAGS, rows=tr), HSMMSpec(K=1, **FIT_KW))
        sc[name] = predictive_scores(res.params, d.data(xs=xs, xc=LAGS, rows=~tr))
        if name in ("+dir K1",) or (name.startswith("+") and len(DIR[asset]) == 1):
            v["gamma_K1"] = dict(zip(xs, res.params.beta_s[0]))
    if len(DIR[asset]) == 1:
        sc["+dir K1"] = sc[f"+{DIR[asset][0][2:]} K1"]
    SCORES[asset] = sc
    B_rows.append(compare(sc, v["test_sessions"], [(m, "Base K1") for m in sc if m != "Base K1"]).assign(underlying=asset))
partB = pd.concat(B_rows, ignore_index=True)
print("Paired by session vs Base K1: mean per-session log-score difference (nats) and whole-session bootstrap 95% interval")
display(partB.pivot(index="model", columns="underlying", values="mean_session_diff").round(2))
display(partB[["underlying"] + SHOW].round(3))
print("Non-switching slopes (all eligible directional factors):")
display(pd.DataFrame({a: v.get("gamma_K1", {}) for a, v in designs.items()}).round(3))

Paired by session vs Base K1: mean per-session log-score difference (nats) and whole-session bootstrap 95% interval


underlying,AVAX,BTC,ETH,HYPE,SOL,XRP
model,,,,,,
+basis K1,0.06,-0.20,-0.03,-3.59,0.24,-0.08
+book K1,-0.26,0.07,-0.06,0.09,-3.01,-0.18
+dir K1,-0.25,-0.54,-0.04,-3.80,-1.94,-0.18
+flow K1,0.00,-0.11,-0.02,-0.08,-0.63,-0.00
+skew K1,-0.01,-0.11,-0.12,-0.09,-0.16,0.10


,underlying,model,vs,n_sessions,mean_session_diff,boot_lo,boot_hi,median_session_diff,sessions_won,loso_min,loso_max,per_bar_x1000,per_bar_nw_t
0,AVAX,+flow K1,Base K1,8,0.002,-0.013,0.018,0.004,0.625,-0.005,0.007,0.010,0.267
1,AVAX,+book K1,Base K1,8,-0.256,-0.774,0.258,-0.355,0.375,-0.478,-0.064,-1.175,-0.828
2,AVAX,+basis K1,Base K1,8,0.064,-0.080,0.193,0.065,0.750,0.028,0.122,0.293,0.672
3,AVAX,+skew K1,Base K1,8,-0.011,-0.066,0.034,0.007,0.500,-0.024,0.009,-0.052,-0.318
4,AVAX,+dir K1,Base K1,8,-0.245,-0.809,0.267,-0.231,0.250,-0.458,-0.016,-1.127,-0.681
5,BTC,+flow K1,Base K1,8,-0.114,-0.548,0.182,0.132,0.625,-0.182,0.095,-0.523,-0.726
6,BTC,+book K1,Base K1,8,0.067,-0.178,0.304,-0.031,0.375,-0.015,0.148,0.307,0.599
7,BTC,+basis K1,Base K1,8,-0.200,-0.748,0.355,-0.185,0.375,-0.407,0.006,-0.917,-0.570
8,BTC,+skew K1,Base K1,8,-0.106,-0.318,0.054,-0.060,0.375,-0.157,-0.012,-0.486,-1.325
9,BTC,+dir K1,Base K1,8,-0.544,-1.748,0.369,-0.281,0.125,-0.830,0.003,-2.499,-1.140


Non-switching slopes (all eligible directional factors):


,AVAX,BTC,ETH,HYPE,SOL,XRP
const,-0.011,-0.007,-0.014,-0.020,0.013,-0.007
D,-0.021,-0.044,-0.062,-0.081,-0.069,-0.016
F_flow,0.001,-0.051,0.004,-0.036,0.052,-0.002
F_book,0.048,0.021,-0.051,-0.026,-0.070,-0.019
F_basis,0.034,0.054,0.022,0.083,0.069,0.053
F_skew,-0.011,-0.031,-0.058,0.027,-0.055,0.028


# Part C — Non-switching dynamic factor model (§21 step 5)

$$
F_t = c + A F_{t-1} + \eta_t
$$

over the asset's eligible factors (directional and state), fitted on the training rows by equation-wise least squares with a small ridge; equation $j$ uses only bars where $F_{j,t}$ and all of $F_{t-1}$ are available and $t-1 \to t$ is one bar (no filling). Reported: eigenvalue moduli of $A$ (persistence), one-step out-of-sample $R^2$ on the test sessions for the full $A$ and for a diagonal $A$ (own lag only: no cross-factor propagation), and stability — $A$ refitted on each half of the training window (relative Frobenius distance, eigenvalue moduli). This establishes whether the latent forces have stable persistence and propagation before switching is introduced.

In [22]:
C_rows, C_A = [], {}
for asset, v in designs.items():
    g, idx = v["g"], v["idx"]
    cols_f = DIR[asset] + STATE[asset]
    if len(cols_f) < 2:
        continue
    F = FS.factors.loc[idx, cols_f].reset_index(drop=True)
    t = pd.to_datetime(g["state_time"], utc=True)
    contiguous = np.r_[False, (t.diff().iloc[1:] == pd.Timedelta(minutes=5)).to_numpy()]
    tr = g["session_date"].isin(TRAIN_DATES).to_numpy()
    rep = dfm.var1_report(F, contiguous, tr, ~tr)
    C_A[asset] = rep
    for n in rep["names"]:
        C_rows.append({"underlying": asset, "factor": n[2:], "own_lag": rep["A"].loc[n, n],
                       "r2_full_A": rep["r2"].loc[n, "full_A"], "r2_diagonal_A": rep["r2"].loc[n, "diagonal_A"],
                       "train_rows": int(rep["train_rows"][n])})
    print(f"{asset}: |eig(A)| {np.round(rep['eig_moduli'], 2)} | halves {np.round(rep['eig_moduli_half1'], 2)} / "
          f"{np.round(rep['eig_moduli_half2'], 2)} | ||A1 - A2|| / ||A|| = {rep['A_half_distance']:.2f} | test pairs {rep['test_rows']}")
partC = pd.DataFrame(C_rows)
print("Own-lag coefficient and one-step out-of-sample R^2 (full A vs diagonal A):")
display(partC.pivot(index="factor", columns="underlying", values=["own_lag", "r2_full_A", "r2_diagonal_A"]).round(2))
if "BTC" in C_A:
    print("BTC: A (row = equation, column = lagged factor)")
    display(C_A["BTC"]["A"].rename(index=lambda s: s[2:], columns=lambda s: s[2:]).round(2))

AVAX: |eig(A)| [0.96 0.88 0.73 0.73 0.69 0.48 0.48 0.31] | halves [0.95 0.81 0.76 0.68 0.43 0.43 0.42 0.18] / [0.96 0.87 0.78 0.72 0.67 0.53 0.49 0.39] | ||A1 - A2|| / ||A|| = 0.36 | test pairs 1702
BTC: |eig(A)| [0.86 0.81 0.74 0.54 0.44 0.3 ] | halves [0.89 0.68 0.68 0.57 0.38 0.27] / [0.87 0.83 0.73 0.52 0.39 0.3 ] | ||A1 - A2|| / ||A|| = 0.29 | test pairs 1360
ETH: |eig(A)| [0.93 0.86 0.79 0.72 0.65 0.51 0.28] | halves [0.93 0.78 0.75 0.75 0.64 0.54 0.21] / [0.94 0.88 0.78 0.66 0.61 0.46 0.24] | ||A1 - A2|| / ||A|| = 0.26 | test pairs 1434
HYPE: |eig(A)| [0.91 0.82 0.73 0.62 0.6  0.51 0.28] | halves [0.92 0.74 0.69 0.63 0.53 0.4  0.23] / [0.9  0.76 0.76 0.71 0.63 0.46 0.29] | ||A1 - A2|| / ||A|| = 0.34 | test pairs 1477
SOL: |eig(A)| [0.88 0.86 0.78 0.74 0.49 0.46 0.3 ] | halves [0.84 0.84 0.73 0.61 0.51 0.33 0.25] / [0.9  0.87 0.82 0.62 0.48 0.48 0.3 ] | ||A1 - A2|| / ||A|| = 0.31 | test pairs 1242
XRP: |eig(A)| [0.93 0.83 0.75 0.69 0.53 0.44 0.26] | halves [0.96 0.87 0.72 0.62 0.

own_lag                               r2_full_A                               r2_diagonal_A                              
underlying      AVAX   BTC   ETH  HYPE   SOL   XRP      AVAX   BTC   ETH  HYPE   SOL   XRP          AVAX   BTC   ETH  HYPE   SOL   XRP
factor                                                                                                                                
activity        0.34   NaN   NaN  0.30   NaN  0.30      0.27   NaN   NaN  0.25   NaN  0.31          0.27   NaN   NaN  0.29   NaN  0.32
basis           0.71  0.45  0.29  0.61  0.38  0.52      0.28  0.26  0.44  0.67  0.16  0.13          0.29  0.27  0.42  0.73  0.19  0.19
book            0.94  0.76  0.81  0.90  0.84  0.81      0.96  0.60  0.71  0.82  0.77  0.83          0.96  0.60  0.71  0.82  0.77  0.84
flow            0.47  0.30  0.51  0.63  0.49  0.45      0.31  0.16  0.37  0.42  0.28  0.21          0.34  0.15  0.36  0.42  0.32  0.21
liquidity       0.71  0.53  0.65  0.77  0.72  0.90      0.69  0.65  0.77  0.80  0.82  0.77          0.71  0.66  0.76  0.83  0.82  0.77
option_level    0.90   NaN  0.93   NaN  0.88   NaN      0.95   NaN  0.93   NaN  0.72   NaN          0.95   NaN  0.93   NaN  0.72   NaN
skew            0.47  0.85  0.77  0.52  0.41  0.69      0.37  0.67  0.80  0.42  0.38  0.54          0.41  0.67  0.80  0.59  0.45  0.57
volatility      0.73  0.81  0.79  0.75  0.77  0.75      0.73  0.56  0.68  0.63  0.70  0.78          0.75  0.58  0.68  0.65  0.70  0.79

BTC: A (row = equation, column = lagged factor)


,flow,book,basis,skew,volatility,liquidity
flow,0.30,-0.01,-0.02,0.02,-0.03,0.02
book,-0.05,0.76,-0.02,-0.03,-0.00,-0.02
basis,0.01,0.01,0.45,0.10,0.03,-0.03
skew,0.04,-0.03,0.02,0.85,-0.02,0.02
volatility,-0.02,-0.06,0.01,-0.03,0.81,-0.05
liquidity,-0.05,-0.04,-0.01,-0.01,0.07,0.53


# Part D — Price-only factor-augmented switching benchmark, $\omega_F = 0$ (§21 step 6)

$$
z_{t+1} = \mu_{R_t} + \beta_{D,R_t} D_t + \gamma_{R_t}^\top F^{dir}_t + \rho^\top L_t + \sigma_{R_t}\xi_{t+1}
$$

with no factor-dynamics evidence in the state inference ($\omega_F = 0$). **HMM first** (§6.3): $K = 1 \dots K_{max}$ is selected **inside the training sessions only** using blocked expanding-window CV, the one-standard-error rule and ≥ 20 episodes per state. The episode-support fit is also restricted to the same training-session universe, so the second-half test sessions are never used to choose $K$. This is done for both the factor model and the Notebook 08 benchmark on the same rows. Then, at the selected $K$ (and at $K = 2$ for description when one regime is selected), the HSMM (negative-binomial durations) is compared with the HMM; explicit durations are kept only if they help (§6.12 E).

Paired session comparisons on the test sessions:
- switching vs the non-switching factor baseline (do the factors work through regimes?);
- switching vs the Notebook 08 benchmark at its own selected $K$ (benchmark 1, §6.13);
- HSMM vs HMM at the same $K$.

In [23]:
HMM = dict(FIT_KW, duration="geometric")
HSMM = dict(FIT_KW, duration="negbin")
sel_rows, K_SEL, D_rows = [], {}, []
for asset, v in designs.items():
    d, tr = v["d"], v["tr"]
    xs_dir = ["const", "D"] + DIR[asset]
    t0 = time.perf_counter()
    for label, xs in [("NB08", ["const", "D"]), ("+dir", xs_dir)]:
        K_sel, tab, _ = select_K(d, K_values=tuple(range(1, K_MAX + 1)), spec_kw=HMM, verbose=False,
                                 data_kw={"xs": xs, "xc": LAGS}, allowed_sessions=TRAIN_DATES)
        sel_rows.append(tab.reset_index().assign(underlying=asset, model=label))
        K_SEL[(asset, label)] = K_sel
    Kd = max(K_SEL[(asset, "+dir")], 2)
    Kb = max(K_SEL[(asset, "NB08")], 2)
    sc = SCORES[asset]
    for name, xs, K, kw in [(f"NB08 HMM K{Kb}", ["const", "D"], Kb, HMM), (f"+dir HMM K{Kd}", xs_dir, Kd, HMM),
                            (f"+dir HSMM K{Kd}", xs_dir, Kd, HSMM)]:
        res = fit(d.data(xs=xs, xc=LAGS, rows=tr), HSMMSpec(K=K, seed=1, **kw))
        sc[name] = predictive_scores(res.params, d.data(xs=xs, xc=LAGS, rows=~tr))
    pairs = [(f"+dir HMM K{Kd}", "+dir K1"), (f"+dir HMM K{Kd}", f"NB08 HMM K{Kb}"), (f"+dir HSMM K{Kd}", f"+dir HMM K{Kd}"),
             (f"NB08 HMM K{Kb}", "Base K1")]
    D_rows.append(compare(sc, v["test_sessions"], pairs).assign(underlying=asset, K_selected_dir=K_SEL[(asset, "+dir")],
                                                                K_selected_nb08=K_SEL[(asset, "NB08")]))
    print(f"{asset}: selected K (HMM) NB08 = {K_SEL[(asset, 'NB08')]}, +dir = {K_SEL[(asset, '+dir')]} "
          f"({time.perf_counter() - t0:.0f}s)")
k_sel = pd.concat(sel_rows, ignore_index=True)
base = k_sel[(k_sel.model == "NB08") & (k_sel.K == 1)].set_index("underlying").oos_logscore
k_sel["oos_vs_NB08_K1"] = k_sel.oos_logscore - k_sel.underlying.map(base)
print("CV out-of-sample log score relative to NB08 at K = 1 (nats, CV test sessions):")
display(k_sel.pivot_table(index=["model", "K"], columns="underlying", values="oos_vs_NB08_K1").round(1))
print("Episodes per state, +dir (support needs >= 20):")
display(k_sel[k_sel.model == "+dir"].pivot(index="K", columns="underlying", values="episodes_per_state"))
partD = pd.concat(D_rows, ignore_index=True)
print("Paired by session (test half):")
display(partD[["underlying"] + SHOW].round(3))

AVAX: selected K (HMM) NB08 = 1, +dir = 1 (79s)
BTC: selected K (HMM) NB08 = 2, +dir = 1 (59s)
ETH: selected K (HMM) NB08 = 1, +dir = 1 (50s)
HYPE: selected K (HMM) NB08 = 1, +dir = 1 (53s)
SOL: selected K (HMM) NB08 = 1, +dir = 1 (63s)
XRP: selected K (HMM) NB08 = 2, +dir = 1 (56s)
CV out-of-sample log score relative to NB08 at K = 1 (nats, CV test sessions):


underlying  AVAX   BTC   ETH  HYPE   SOL   XRP
model K                                       
+dir  1     -2.5  -0.1   0.1  -3.6   5.2  -6.9
      2    -12.4  -3.6  -2.1  -6.0   7.5 -10.8
      3    -23.3  -0.6  -2.6  -2.7  -3.5 -17.0
      4    -29.3 -11.5 -25.0 -15.5 -13.1 -23.8
NB08  1      0.0   0.0   0.0   0.0   0.0   0.0
      2     -3.1   5.1   4.1   1.6  -1.0   3.0
      3    -10.9   4.0  -2.0  -0.1  -0.7   2.1
      4    -12.6  -2.6  -5.3  -3.0  -1.7 -13.2

Episodes per state, +dir (support needs >= 20):


underlying,AVAX,BTC,ETH,HYPE,SOL,XRP
K,,,,,,
1,None,None,None,None,None,None
2,"[21, 34]","[27, 27]","[37, 42]","[27, 27]","[51, 48]","[49, 51]"
3,"[10, 39, 26]","[188, 21, 191]","[23, 77, 84]","[66, 112, 41]","[21, 85, 93]","[16, 53, 58]"
4,"[89, 286, 67, 212]","[18, 193, 13, 188]","[36, 237, 256, 44]","[20, 160, 50, 158]","[39, 106, 28, 79]","[143, 24, 104, 50]"


Paired by session (test half):


,underlying,model,vs,n_sessions,mean_session_diff,boot_lo,boot_hi,median_session_diff,sessions_won,loso_min,loso_max,per_bar_x1000,per_bar_nw_t
0,AVAX,+dir HMM K2,+dir K1,8,-0.899,-1.809,-0.123,-0.920,0.250,-1.118,-0.509,-4.131,-1.979
1,AVAX,+dir HMM K2,NB08 HMM K2,8,-1.121,-1.875,-0.343,-1.053,0.125,-1.461,-0.894,-5.156,-2.504
2,AVAX,+dir HSMM K2,+dir HMM K2,8,-0.584,-1.120,-0.021,-0.640,0.250,-0.781,-0.419,-2.687,-1.633
3,AVAX,NB08 HMM K2,Base K1,8,-0.022,-1.045,0.916,-0.264,0.375,-0.328,0.367,-0.103,-0.044
4,BTC,+dir HMM K2,+dir K1,8,0.952,-0.581,2.436,1.175,0.625,0.416,1.503,4.372,1.647
5,BTC,+dir HMM K2,NB08 HMM K2,8,-1.146,-2.663,-0.140,-0.724,0.125,-1.422,-0.445,-5.261,-1.878
6,BTC,+dir HSMM K2,+dir HMM K2,8,0.235,0.054,0.401,0.258,0.750,0.175,0.299,1.078,1.642
7,BTC,NB08 HMM K2,Base K1,8,1.553,-0.108,3.144,1.060,0.875,1.039,2.125,7.134,2.664
8,ETH,+dir HMM K2,+dir K1,8,-0.898,-1.799,0.028,-1.010,0.250,-1.255,-0.596,-4.126,-1.405
9,ETH,+dir HMM K2,NB08 HMM K2,8,-0.675,-1.935,0.210,-0.116,0.375,-0.904,-0.128,-3.099,-1.384


# Part E — States of the benchmark (full sample, HMM, $K = \max(K_{sel}, 2)$)

Description only (§7). States ordered by $\beta_D$. A directional factor discriminates regimes only if its slope differs between states; with similar slopes it helps price but not through regimes.

In [24]:
state_rows, ep_rows = [], []
for asset, v in designs.items():
    d = v["d"]
    xs = ["const", "D"] + DIR[asset]
    K = max(K_SEL[(asset, "+dir")], 2)
    data = d.data(xs=xs, xc=LAGS)
    res = fit(data, HSMMSpec(K=K, n_init=4, max_iter=300, seed=1, duration="geometric"))
    p = res.params
    for k in range(K):
        row = {"underlying": asset, "K": K, "state": k, "occupancy": float(res.gamma[:, k].mean()), "sigma": p.sigma[k],
               "mean_duration": float(p.mean_duration()[k]), "nu": p.nu}
        row.update({f"b_{n}": p.beta_s[k, j] for j, n in enumerate(xs)})
        state_rows.append(row)
    ep = pd.DataFrame(episodes(res.gamma, data))
    ep_rows.append(ep.groupby("state").agg(episodes=("length", "size"), median_bars=("length", "median")).assign(underlying=asset).reset_index())
states = pd.DataFrame(state_rows)
display(states.round(3))
display(pd.concat(ep_rows).set_index(["underlying", "state"]))

,underlying,K,state,occupancy,sigma,mean_duration,nu,b_const,b_D,b_F_flow,b_F_book,b_F_basis,b_F_skew
0,AVAX,2,0,0.387,0.610,7.389,5.0,-0.070,-0.101,0.028,0.039,0.003,-0.050
1,AVAX,2,1,0.613,0.945,12.079,5.0,0.069,0.011,-0.032,0.009,0.062,0.033
2,BTC,2,0,0.704,0.688,8.822,5.0,-0.031,-0.147,-0.033,0.010,0.023,0.012
3,BTC,2,1,0.296,1.158,3.560,5.0,0.151,0.283,-0.056,0.077,0.060,-0.008
4,ETH,2,0,0.650,0.707,11.291,5.0,-0.027,-0.118,0.045,-0.059,0.047,-0.070
5,ETH,2,1,0.350,1.045,5.540,5.0,0.029,0.084,-0.107,0.024,-0.023,0.027
6,HYPE,2,0,0.514,0.756,13.320,6.0,0.086,-0.108,-0.090,-0.014,0.109,-0.019
7,HYPE,2,1,0.486,0.964,11.094,6.0,-0.101,-0.034,0.040,-0.024,-0.092,0.050
8,SOL,2,0,0.548,0.709,8.957,5.0,-0.048,-0.137,-0.084,-0.046,0.028,-0.017
9,SOL,2,1,0.452,0.987,6.871,5.0,0.088,0.020,0.171,0.034,0.116,-0.078


episodes  median_bars
underlying state                       
AVAX       0           117          7.0
           1           122         17.0
BTC        0           173         11.0
           1           175          2.0
ETH        0           130         15.0
           1           137          4.0
HYPE       0           104         16.0
           1           111         10.0
SOL        0           156          9.0
           1           167          6.0
XRP        0           105         26.0
           1           106          2.0

# Part F — Conditional sign-destruction null (§6.10, first of two)

Random signs on the real return magnitudes: the directional relation between returns and the observed market state is destroyed while the volatility path, the features and the missing-data pattern are kept. Return-derived model inputs are rebuilt from the simulated returns (index path, session VWAP and its displacement, $\sigma^c$, $z$, lags); the return-derived factor inputs (realized volatility, range) depend only on $|r|$ and are unchanged by sign flips. Statistic: separation of the state slopes over $D$ and the directional factors at $K = 2$ (HMM),

$$
\Delta = \bigl\|\boldsymbol\beta_1 - \boldsymbol\beta_0\bigr\|_2, \qquad \boldsymbol\beta_k = (\beta_{D,k}, \gamma_k)
$$

One-sided $p = (1 + \#\{\Delta^* \ge \hat\Delta\})/(1 + N)$. **Passing this null alone is not confirmatory** (§6.10): the end-to-end generative null (non-switching dynamic factor model with stochastic volatility, simulated through the full feature pipeline) is run with the joint SDF model (§21 step 9).

In [25]:
def slope_sep(design, xs):
    res = fit(design.data(xs=xs, xc=LAGS), HSMMSpec(K=2, seed=3, **HMM))
    b = res.params.beta_s[:, 1:]
    return float(np.linalg.norm(b[1] - b[0]))


null_rows = []
for asset in NULL_ASSETS:
    if asset not in designs:
        continue
    v = designs[asset]
    d, g = v["d"], v["g"]
    xs = ["const", "D"] + DIR[asset]
    E = FS.factors.loc[v["idx"], DIR[asset]].reset_index(drop=True)
    t0 = time.perf_counter()
    real = slope_sep(d, xs)
    rng = np.random.default_rng(sum(map(ord, asset)))
    draws = []
    for _ in range(NULL_PATHS):
        r_star = simulate_returns(g, g["f_clock"].to_numpy(), "sign", rng)
        dn = build_design(g, g["f_clock"].to_numpy(), returns=r_star, extra=E if len(E.columns) else None, rng=rng)
        draws.append(slope_sep(dn, xs))
    draws = np.array(draws)
    null_rows.append({"underlying": asset, "separation_real": real, "null_p50": np.median(draws),
                      "null_p95": np.quantile(draws, 0.95), "p_value": p_value(real, draws),
                      "seconds": time.perf_counter() - t0})
null_test = pd.DataFrame(null_rows)
display(null_test.round(3))

,underlying,separation_real,null_p50,null_p95,p_value,seconds
0,BTC,0.317,0.178,0.335,0.15,27.343
1,XRP,0.260,0.243,0.395,0.45,25.658
2,ETH,0.234,0.224,0.310,0.50,24.040
3,SOL,0.299,0.163,0.322,0.15,23.171


## Reading the results (v3.2)

| Result | v3.2 consequence |
|---|---|
| A: family ineligible (coverage or loading stability) | Excluded from the switching model until the data support a stable basis (§6.8). |
| B: a family improves on Base with a bootstrap interval above 0 and most sessions won | The factor helps price without regimes. |
| C: full $A$ beats diagonal $A$ out of sample, eigenvalues stable across halves | Stable cross-factor propagation: a basis for regime-specific $A_R$ (next module). |
| C: $A$ unstable between halves | Factor dynamics not ready for switching (§6.12). |
| D: switching beats the non-switching factor baseline and the Notebook 08 benchmark on sessions | The price-only factor benchmark (§6.13 benchmark 2) carries regime information. |
| D: HSMM ≈ HMM | Duration dependence not established (§6.12). |
| F: $p \le 0.05$ | Passes the conditional null; still needs the generative null (§6.10). |

All of this is discovery evidence at ≈ 15 sessions (§6.14).

In [26]:
nb09_report = {"factor_summary": FS.summary, "loadings": FS.loadings, "baseline": partB, "dfm": partC,
               "k_selection": k_sel, "benchmark": partD, "states": states, "null_test": null_test}
for k, v in nb09_report.items():
    print(f"  {k:<15} {len(v):>6,} rows")
con.close()
print("DuckDB connection closed.")

  factor_summary      48 rows
  loadings           126 rows
  baseline            30 rows
  dfm                 42 rows
  k_selection         48 rows
  benchmark           24 rows
  states              12 rows
  null_test            4 rows
DuckDB connection closed.
